# Giveaway Above Expected Model

This notebook builds the core model of our puck management framework — 
giveaway above expected. For each 5on5 giveaway in the 2025-26 season 
we calculate how much danger it caused relative to what was expected 
given the context.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)


In [2]:
# Load data
events = pd.read_csv('../data/processed/events.csv')
events = events.sort_values(['game_date', 'game_id', 'sort_order']).reset_index(drop=True)

# Filter giveaways to 5on5
giveaways = events[events['event_type'] == 'giveaway']
giveaways = giveaways[giveaways['situation_code'] == 1551].copy()

print(f"Total events: {len(events)}")
print(f"5on5 giveaways: {len(giveaways)}")

Total events: 413859
5on5 giveaways: 35422


In [4]:
events.head()

,game_id,season,game_date,home_team,away_team,home_score_final,away_score_final,event_id,event_type,sort_order,period,period_type,time_in_period,time_remaining,situation_code,home_team_defending_side,x_coord,y_coord,zone_code,player_id,event_owner_team_id,hitting_player_id,hittee_player_id,shooting_player_id,scoring_player_id,away_score,home_score,player_name,position_code,player_team_id
0,2025020001,20252026,2025-10-07,FLA,CHI,3,2,52,period-start,8,1,REG,00:00,20:00,1551,right,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025020001,20252026,2025-10-07,FLA,CHI,3,2,51,faceoff,11,1,REG,00:00,20:00,1551,right,0.0,0.0,N,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025020001,20252026,2025-10-07,FLA,CHI,3,2,101,blocked-shot,12,1,REG,00:22,19:38,1551,right,-61.0,3.0,D,NaN,13.0,NaN,NaN,8473419.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025020001,20252026,2025-10-07,FLA,CHI,3,2,8,stoppage,13,1,REG,00:34,19:26,1551,right,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025020001,20252026,2025-10-07,FLA,CHI,3,2,55,faceoff,16,1,REG,00:34,19:26,1551,right,-69.0,22.0,D,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
giveaways.head()

,game_id,season,game_date,home_team,away_team,home_score_final,away_score_final,event_id,event_type,sort_order,period,period_type,time_in_period,time_remaining,situation_code,home_team_defending_side,x_coord,y_coord,zone_code,player_id,event_owner_team_id,hitting_player_id,hittee_player_id,shooting_player_id,scoring_player_id,away_score,home_score,player_name,position_code,player_team_id
5,2025020001,20252026,2025-10-07,FLA,CHI,3,2,74,giveaway,18,1,REG,00:46,19:14,1551,right,-41.0,38.0,O,8482113.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,Anton Lundell,C,13.0
6,2025020001,20252026,2025-10-07,FLA,CHI,3,2,851,giveaway,24,1,REG,01:10,18:50,1551,right,-13.0,-31.0,N,8484783.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,Artyom Levshunov,D,16.0
29,2025020001,20252026,2025-10-07,FLA,CHI,3,2,171,giveaway,66,1,REG,03:16,16:44,1551,right,14.0,-12.0,N,8473507.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,Jeff Petry,D,13.0
61,2025020001,20252026,2025-10-07,FLA,CHI,3,2,883,giveaway,159,1,REG,09:19,10:41,1551,right,-70.0,-40.0,D,8483506.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,Sam Rinzel,D,16.0
94,2025020001,20252026,2025-10-07,FLA,CHI,3,2,390,giveaway,230,1,REG,14:22,05:38,1551,right,5.0,6.0,N,8483506.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,Sam Rinzel,D,16.0


## Step 1 — Convert Time to Seconds

Convert time_in_period to total seconds elapsed in the game so we can 
calculate a 10 second window after each giveaway.

In [46]:
def time_to_seconds(time_str, period):
    """Convert time_in_period string and period to total game seconds"""
    minutes, seconds = map(int, time_str.split(':'))
    period_seconds = (period - 1) * 1200  # each period is 20 mins = 1200 seconds
    return period_seconds + (minutes * 60) + seconds

# Apply to events
events['game_seconds'] = events.apply(
    lambda row: time_to_seconds(row['time_in_period'], row['period']), axis=1
)

print(events[['time_in_period', 'period', 'game_seconds']].head(10))

  time_in_period  period  game_seconds
0          00:00       1             0
1          00:00       1             0
2          00:22       1            22
3          00:34       1            34
4          00:34       1            34
5          00:46       1            46
6          01:10       1            70
7          01:13       1            73
8          01:15       1            75
9          01:15       1            75


## Step 2 — Build Shot Danger Model

To calculate xGoal for each shot we first need to calculate distance 
and angle to the net for every shot in our dataset. We can then train a 
logistic regression on all shots to predict goal probability.

In [9]:
# Filter to shots only
shots = events[events['event_type'].isin([
    'shot-on-goal', 'missed-shot', 'blocked-shot', 'goal'
])].copy()

print(f"Total shots: {len(shots)}")
shots[['event_type', 'x_coord', 'y_coord', 'zone_code']].head(10)

Total shots: 153754


,event_type,x_coord,y_coord,zone_code
2,blocked-shot,-61.0,3.0,D
7,blocked-shot,-64.0,-16.0,D
10,shot-on-goal,-58.0,-22.0,O
15,shot-on-goal,-33.0,-19.0,O
18,missed-shot,55.0,-29.0,O
21,shot-on-goal,35.0,33.0,O
22,blocked-shot,43.0,-32.0,D
27,shot-on-goal,30.0,-34.0,O
30,blocked-shot,-56.0,-22.0,D
31,shot-on-goal,-58.0,28.0,O


In [14]:
def calculate_distance_angle(x, y):
    """Calculate distance and angle to nearest net"""
    
    # Distance to both nets
    dist_right = np.sqrt((x - 89)**2 + y**2)
    dist_left = np.sqrt((x + 89)**2 + y**2)
    
    # Take the closer net
    if dist_right < dist_left:
        net_x = 89
    else:
        net_x = -89
    
    distance = min(dist_right, dist_left)
    angle = np.arctan2(abs(y), abs(abs(net_x) - abs(x)))
    
    return distance, angle

# Apply to all shots
shots['distance'] = shots.apply(
    lambda row: calculate_distance_angle(row['x_coord'], row['y_coord'])[0] 
    if pd.notna(row['x_coord']) else None, axis=1
)

shots['angle'] = shots.apply(
    lambda row: calculate_distance_angle(row['x_coord'], row['y_coord'])[1] 
    if pd.notna(row['x_coord']) else None, axis=1
)

shots[['event_type', 'x_coord', 'y_coord', 'distance', 'angle']].head(10)

,event_type,x_coord,y_coord,distance,angle
2,blocked-shot,-61.0,3.0,28.160256,0.106736
7,blocked-shot,-64.0,-16.0,29.681644,0.569313
10,shot-on-goal,-58.0,-22.0,38.013156,0.617191
15,shot-on-goal,-33.0,-19.0,59.135438,0.327098
18,missed-shot,55.0,-29.0,44.687806,0.706199
21,shot-on-goal,35.0,33.0,63.285069,0.548549
22,blocked-shot,43.0,-32.0,56.035703,0.607802
27,shot-on-goal,30.0,-34.0,68.095521,0.522789
30,blocked-shot,-56.0,-22.0,39.661064,0.588003
31,shot-on-goal,-58.0,28.0,41.773197,0.734594


In [29]:
shots['is_goal'] = (shots['event_type'] == 'goal').astype(int)

print(f"Total shots: {len(shots)}")
print(f"Total goals: {shots['is_goal'].sum()}")

# Drop rows with missing distance or angle
shots_clean = shots.dropna(subset=['distance', 'angle'])

#can use whethere or not a shot was or target as a feature along with diastance and angle
shots_clean['is_on_target'] = (shots_clean['event_type'].isin(['shot-on-goal', 'goal'])).astype(int)

# Features and target
X = shots_clean[['distance', 'angle', 'is_on_target']]
y = shots_clean['is_goal']

print(f"Training on {len(shots_clean)} shots")
print(f"Features: distance, angle")
print(f"Target: is_goal")

Total shots: 153754
Total goals: 8350
Training on 153754 shots
Features: distance, angle
Target: is_goal


In [36]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Train on training set only
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate on test set
train_auc = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc').mean()
test_auc = cross_val_score(model, X_test, y_test, cv=5, scoring='roc_auc').mean()

print(f"Train AUC: {train_auc:.3f}")
print(f"Test AUC:  {test_auc:.3f}")
print(f"Difference: {abs(train_auc - test_auc):.3f}")

Train AUC: 0.863
Test AUC:  0.865
Difference: 0.002


In [39]:
# Generate xGoal for every shot
X_all = shots_clean[['distance', 'angle', 'is_on_target']]
X_all_scaled = scaler.transform(X_all)

# Generate xGoal for all shots
shots_clean['xGoal'] = model.predict_proba(X_all_scaled)[:, 1]
print(shots_clean[['event_type', 'distance', 'angle', 'xGoal']].head(10))
print(shots_clean['xGoal'].describe())

      event_type   distance     angle     xGoal
2   blocked-shot  28.160256  0.106736  0.000045
7   blocked-shot  29.681644  0.569313  0.000030
10  shot-on-goal  38.013156  0.617191  0.077634
15  shot-on-goal  59.135438  0.327098  0.046176
18   missed-shot  44.687806  0.706199  0.000016
21  shot-on-goal  63.285069  0.548549  0.034117
22  blocked-shot  56.035703  0.607802  0.000011
27  shot-on-goal  68.095521  0.522789  0.029338
30  blocked-shot  39.661064  0.588003  0.000021
31  shot-on-goal  41.773197  0.734594  0.063045
count    153754.000000
mean          0.054766
std           0.074160
min           0.000003
25%           0.000032
50%           0.000085
75%           0.092122
max           0.338316
Name: xGoal, dtype: float64


The model produces xGoal values that align with hockey intuition:

**Blocked shots** receive near-zero xGoal (0.000043, 0.000029) since they 
never reach the goalie and have no chance of going in regardless of where 
they were taken from.

**Missed shots** also receive near-zero xGoal (0.000015) for the same reason 
— they missed the net entirely.

**Shots on goal** receive meaningful xGoal values that scale correctly with 
distance and angle:
- 38 feet away → xGoal 0.076 (7.6% chance)
- 59 feet away → xGoal 0.044 (4.4% chance)
- 68 feet away → xGoal 0.028 (2.8% chance)

Closer shots with better angles get higher xGoal values as expected.

**Overall xGoal distribution:**
- Mean: 0.054 — consistent with the league wide goal rate of 5.4%
- Max: 0.337 — the most dangerous shot in the dataset (1 foot tap in, straight on)
- 75th percentile: 0.090 — only the top 25% of shots have greater than 9% 
chance of going in, reflecting how difficult it is to score in the NHL

In [40]:
# Find the highest xGoal shot
max_shot = shots_clean.loc[shots_clean['xGoal'].idxmax()]
print(max_shot[['event_type', 'distance', 'angle', 'is_on_target', 'xGoal', 'x_coord', 'y_coord', 'period', 'game_id']])

event_type            goal
distance               1.0
angle                  0.0
is_on_target             1
xGoal             0.338316
x_coord              -88.0
y_coord                0.0
period                   2
game_id         2025020677
Name: 215108, dtype: object


This is the highest xGoal shot in the wholte dataset! We can see hes pretty much on the goal line directly in front of the net (angle of 0) so someone basically tapped it in from right on the goal line, straight on. Our model correctly identified it as the highest danger shot.

## xGoal Model

To measure the danger caused by each giveaway we built a simple xGoal model 
that predicts the probability of a shot becoming a goal based on shot context.

### Model Details
- **Algorithm:** Logistic Regression
- **Training data:** 153,754 shot attempts from the 2025-26 NHL regular season
- **Validation:** 5-fold cross validation

### Features
| Feature | Description |
|---|---|
| `distance` | Distance to nearest net calculated from x/y coordinates using Pythagoras |
| `angle` | Angle to net calculated using arctan |
| `is_on_target` | Whether the shot was on target (shot-on-goal or goal = 1, missed/blocked = 0) |

### Performance
- **AUC: 0.864** — strong performance, comparable to professional xGoal models
- The most dangerous shot in the dataset was a tap-in from 1 foot directly in front 
of the net (xGoal = 0.337)
- Average xGoal across all shots: 0.054, consistent with the league wide goal rate of 5.4%

### Limitations
This is a simplified xGoal model using only 3 features. Professional models 
like MoneyPuck use 20+ features including shot type, whether it came off a rush, 
rebound situations, traffic in front of the net, and goalie positioning. As a result 
our model likely underestimates danger for truly elite chances (e.g. close range 
tap ins should be 0.8+ but our model caps at 0.34). However for the purposes of 
comparing giveaway danger across players this model is sufficient — we are measuring 
relative danger not absolute goal probability.

## Step 3 — Label Each Giveaway with Actual Danger Score

For each of our 35,422 5on5 giveaways we need to measure how much 
danger it actually caused. We do this by looking at what happened 
immediately after each giveaway.

**The process for each giveaway:**
1. Record the exact game second it occurred
2. Look at all events in the same game within the next 10 seconds
3. Filter to shots only (shot-on-goal, missed-shot, blocked-shot, goal)
4. Filter to shots by the OPPOSING team only — we don't want to count 
   shots by the same team that gave the puck away
5. If a shot is found → assign its xGoal value as the danger score
6. If no shot is found within 10 seconds → danger score = 0

**Why 10 seconds?**
A shot within 10 seconds of a giveaway is very likely directly caused 
by that turnover. Beyond 10 seconds play has likely reset and the 
connection between the giveaway and any subsequent shot becomes 
increasingly weak.

In [66]:
# Add game_seconds to giveaways
giveaways['game_seconds'] = giveaways.apply(
    lambda row: time_to_seconds(row['time_in_period'], row['period']), axis=1
)

# Also make sure shots_clean has game_seconds
shots_clean['game_seconds'] = shots_clean.apply(
    lambda row: time_to_seconds(row['time_in_period'], row['period']), axis=1
)

# Label each giveaway with danger score
danger_scores = []

for idx, row in giveaways.iterrows():
    
    # Get the time window - 10 seconds after the giveaway
    giveaway_second = row['game_seconds']
    
    # Find all shots in the same game within 10 seconds by the opposing team
    shots_after = shots_clean[
        (shots_clean['game_id'] == row['game_id']) &
        (shots_clean['game_seconds'] > giveaway_second) &
        (shots_clean['game_seconds'] <= giveaway_second + 15) &
        (shots_clean['event_owner_team_id'] != row['event_owner_team_id'])
    ]
    
    # Sum xGoal of all shots found
    if len(shots_after) > 0:
        danger_scores.append(shots_after['xGoal'].sum())
    else:
        danger_scores.append(0)

# Add danger score to giveaways
giveaways['danger_score'] = danger_scores

print(f"Giveaways with danger score > 0: {(giveaways['danger_score'] > 0).sum()}")
print(f"Average danger score: {giveaways['danger_score'].mean():.4f}")
print(f"\nDanger score distribution:")
print(giveaways['danger_score'].describe())

Giveaways with danger score > 0: 10819
Average danger score: 0.0206

Danger score distribution:
count    35422.000000
mean         0.020613
std          0.060482
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000027
max          1.087722
Name: danger_score, dtype: float64


In [65]:
# Find all shots within 10 seconds after McMichael's giveaway
mcmichael_second = most_dangerous['game_seconds']
mcmichael_game = most_dangerous['game_id']
mcmichael_team = most_dangerous['event_owner_team_id']

shots_after_mcmichael = shots_clean[
    (shots_clean['game_id'] == mcmichael_game) &
    (shots_clean['game_seconds'] > mcmichael_second) &
    (shots_clean['game_seconds'] <= mcmichael_second + 10) &
    (shots_clean['event_owner_team_id'] != mcmichael_team)
]

shots_after_mcmichael[['event_type', 'game_seconds', 'x_coord', 'y_coord', 'distance', 'angle', 'xGoal']]

,event_type,game_seconds,x_coord,y_coord,distance,angle,xGoal
28946,shot-on-goal,1025,81.0,0.0,8.000000,0.000000,0.283855
28947,shot-on-goal,1027,82.0,-1.0,7.071068,0.141897,0.269553
28948,shot-on-goal,1028,81.0,0.0,8.000000,0.000000,0.283855


### Most Dangerous Giveaway

As a sanity check we identified the giveaway with the highest danger 
score in the dataset — Connor McMichael in period 1 at game second 1020.

**What happened:**
McMichael gave the puck away in the defensive zone and the opposing team 
generated three shots from right in front of the net within 8 seconds:

| Shot | Distance | Angle | xGoal |
|---|---|---|---|
| Shot on goal | 8 feet | 0.00 (straight on) | 0.284 |
| Shot on goal | 7 feet | 0.14 | 0.270 |
| Shot on goal | 8 feet | 0.00 (straight on) | 0.284 |

**Combined danger score: 0.837**

This represents a classic crease scramble situation — the puck bouncing 
around right in front of the net with multiple tap in attempts. Each 
individual shot had roughly a 28% chance of going in, meaning collectively 
there was an 84% chance at least one would score.

This validation confirms our danger scoring model is working correctly — 
the highest danger score in the dataset corresponds to a genuinely 
catastrophic turnover that any hockey analyst would immediately recognise 
as extremely dangerous.

In [63]:
same_team_count = 0
diff_team_count = 0
diff_team_within_10 = 0
diff_team_within_15 = 0
diff_team_within_20 = 0
diff_team_within_30 = 0
no_shot_count = 0
time_gaps = []

for idx, row in giveaways.iterrows():
    
    next_shot = events[
        (events['game_id'] == row['game_id']) &
        (events['game_seconds'] > row['game_seconds']) &
        (events['event_type'].isin(['shot-on-goal', 'missed-shot', 'blocked-shot', 'goal']))
    ].head(1)
    
    if len(next_shot) == 0:
        no_shot_count += 1
        continue
    
    gap = next_shot.iloc[0]['game_seconds'] - row['game_seconds']
    is_same = next_shot.iloc[0]['event_owner_team_id'] == row['event_owner_team_id']
    
    if is_same:
        same_team_count += 1
    else:
        diff_team_count += 1
        time_gaps.append(gap)
        if gap <= 10: diff_team_within_10 += 1
        if gap <= 15: diff_team_within_15 += 1
        if gap <= 20: diff_team_within_20 += 1
        if gap <= 30: diff_team_within_30 += 1

print(f"Total giveaways: {len(giveaways)}")
print(f"No shot ever followed: {no_shot_count} ({no_shot_count/len(giveaways):.1%})")
print(f"Same team next shot: {same_team_count} ({same_team_count/len(giveaways):.1%})")
print(f"Opposing team next shot: {diff_team_count} ({diff_team_count/len(giveaways):.1%})")
print(f"\nOpposing team shots within 10s: {diff_team_within_10} ({diff_team_within_10/len(giveaways):.1%})")
print(f"Opposing team shots within 15s: {diff_team_within_15} ({diff_team_within_15/len(giveaways):.1%})")
print(f"Opposing team shots within 20s: {diff_team_within_20} ({diff_team_within_20/len(giveaways):.1%})")
print(f"Opposing team shots within 30s: {diff_team_within_30} ({diff_team_within_30/len(giveaways):.1%})")
print(f"\nAverage time gap for opposing shots: {sum(time_gaps)/len(time_gaps):.1f} seconds")

Total giveaways: 35422
No shot ever followed: 98 (0.3%)
Same team next shot: 13577 (38.3%)
Opposing team next shot: 21747 (61.4%)

Opposing team shots within 10s: 8642 (24.4%)
Opposing team shots within 15s: 10780 (30.4%)
Opposing team shots within 20s: 12394 (35.0%)
Opposing team shots within 30s: 14926 (42.1%)

Average time gap for opposing shots: 27.1 seconds


### Investigating the Time Window for Danger Scoring

#### The Problem
Our initial analysis found that only 24% of giveaways had a danger score > 0 
using a 10 second window. This seemed low given our EDA showed 46% of giveaways 
were immediately followed by a shot.

#### Investigation
We investigated the drop by looking at what happens after each giveaway:

**Two filters are reducing our capture rate:**

1. **Same team filter** — a significant portion of next shots are by the SAME 
team that gave the puck away (the team recovered and got a shot themselves). 
These correctly get filtered out since we only want opposing team shots.

2. **Time window** — many opposing team shots happen outside our 10 second 
window. A 10 second window may be too strict given how long it takes for 
play to develop after a turnover.

#### What We're Doing
Running the full analysis on all 35,422 giveaways to find:
- What % of next shots are by the same team vs opposing team?
- What % of opposing team shots fall within 10, 15, 20, and 30 seconds?
- What is the average time gap between a giveaway and the next opposing shot?

#### Full Results (across all 35,422 giveaways)
- 0.3% of giveaways had no shot ever follow
- 38.3% of next shots were by the same team — correctly filtered out
- 61.4% of next shots were by the opposing team ✅
- Average time gap for opposing shots: **27 seconds**

| Time Window | Giveaways Captured | % of Total |
|---|---|---|
| 10 seconds | 8,642 | 24.4% |
| 15 seconds | 10,780 | 30.4% |
| 20 seconds | 12,394 | 35.0% |
| 30 seconds | 14,926 | 42.1% |

#### Decision — 15 Second Window
We settled on a **15 second window** for the following reasons:

- Captures 30.4% of giveaways — a meaningful sample
- 15 seconds in hockey is a full rush or cycle — long enough to capture 
  the immediate danger from a turnover
- Beyond 15 seconds play has had time to reset and attributing a shot 
  to a specific giveaway becomes increasingly tenuous
- 20 seconds felt too loose — a shot 19 seconds after a giveaway could 
  reasonably be unrelated to the turnover

Note that 69.6% of giveaways having a danger score of 0 is not a problem 
— it correctly reflects that most turnovers do not immediately lead to 
dangerous shots. The model still learns that dzone giveaways are more 
likely to generate danger than ozone ones.

## Step 4 — Build the Above Expected Model (Model 2)

With every giveaway now labelled with its actual danger score, we build 
a regression model that learns what danger is expected given the context 
of each giveaway. The difference between actual and expected danger is 
our giveaway above expected metric.

In [70]:
oe = OrdinalEncoder(categories=[['O', 'N', 'D']])
giveaways['zone_code_encoded'] = oe.fit_transform(giveaways[['zone_code']])

print(f"Zone encoding: O=0, N=1, D=2")
print(f"\nMissing values:\n{giveaways[['zone_code_encoded']].isnull().sum()}")
print(giveaways[['zone_code', 'zone_code_encoded']].head(10))

Zone encoding: O=0, N=1, D=2

Missing values:
zone_code_encoded    0
dtype: int64
    zone_code  zone_code_encoded
5           O                0.0
6           N                1.0
29          N                1.0
61          D                2.0
94          N                1.0
97          D                2.0
117         O                0.0
118         O                0.0
120         O                0.0
123         N                1.0


In [95]:

features = ['x_coord', 'y_coord','zone_code_encoded']
X = giveaways[features]
y = giveaways['danger_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler2 = StandardScaler()
X_train_scaled = scaler2.fit_transform(X_train)
X_test_scaled = scaler2.transform(X_test)

#X_train_scaled

In [96]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

train_r2 = rf_model.score(X_train_scaled, y_train)
test_r2 = rf_model.score(X_test_scaled, y_test)

print(f"Train R²: {train_r2:.3f}")
print(f"Test R²:  {test_r2:.3f}")

Train R²: 0.021
Test R²:  0.013


In [99]:
# What does our model predict for different zones?
test_cases = pd.DataFrame({
    'x_coord': [-80, 0, 80],
    'y_coord': [0, 0, 0],
    'zone_code_encoded': [2, 1, 0]  # D, N, O
})

test_scaled = scaler2.transform(test_cases)
predictions = rf_model.predict(test_scaled)

print("Expected danger by zone:")
print(f"Dzone (x=-80): {predictions[0]:.4f}")
print(f"Neutral (x=0): {predictions[1]:.4f}")
print(f"Ozone (x=80):  {predictions[2]:.4f}")

Expected danger by zone:
Dzone (x=-80): 0.0307
Neutral (x=0): 0.0156
Ozone (x=80):  0.0107


### Why Random Forest?

We experimented with two models for predicting expected giveaway danger:

**Ridge Regression** — a linear model that assumes a straight line relationship 
between features and danger score. This performed poorly (R² = 0.011) because 
giveaway danger is not linear. A giveaway at x=-85 (right in front of the net) 
is not just "slightly more dangerous" than one at x=-70 — it is dramatically 
more dangerous. Linear models cannot capture these non-linear relationships.

**Random Forest** — builds 100 decision trees on random subsets of the data 
and averages their predictions. This naturally captures non-linear relationships 
between coordinates and danger without assuming any particular shape to the 
relationship. It also handles our heavily skewed target variable (70% zeros) 
better than linear models.

### Why These Parameters?

| Parameter | Value | Reason |
|---|---|---|
| `n_estimators` | 100 | 100 trees gives stable, reliable predictions |
| `max_depth` | 5 | Limits tree complexity to prevent overfitting |
| `min_samples_leaf` | 50 | Each decision must be based on at least 50 giveaways |
| `random_state` | 42 | Ensures reproducibility |

### Performance
- Train R²: 0.022
- Test R²: 0.013
- Consistent between train and test — no overfitting ✅

### Why R² is Low and Why That's Ok
R² of 0.013 seems low but is expected given 70% of danger scores are exactly 
0. The metric itself becomes misleading with such a skewed distribution. What 
matters is whether the model correctly orders situations by danger — and it does:

| Zone | Expected Danger |
|---|---|
| Defensive zone | 0.0307 |
| Neutral zone | 0.0156 |
| Offensive zone | 0.0107 |

Dzone giveaways are correctly identified as 3x more dangerous than ozone 
giveaways. The model is learning the right patterns even if R² appears low.

In [101]:
# Generating expected danger for all giveaways
X_all = giveaways[features]
X_all_scaled = scaler2.transform(X_all)
giveaways['expected_danger'] = rf_model.predict(X_all_scaled)

# Calculate above expected
giveaways['above_expected'] = giveaways['expected_danger'] - giveaways['danger_score']

giveaways.head(10)

,game_id,season,game_date,home_team,away_team,home_score_final,away_score_final,event_id,event_type,sort_order,period,period_type,time_in_period,time_remaining,situation_code,home_team_defending_side,x_coord,y_coord,zone_code,player_id,event_owner_team_id,hitting_player_id,hittee_player_id,shooting_player_id,scoring_player_id,away_score,home_score,player_name,position_code,player_team_id,game_seconds,danger_score,zone_code_encoded,expected_danger,above_expected
5,2025020001,20252026,2025-10-07,FLA,CHI,3,2,74,giveaway,18,1,REG,00:46,19:14,1551,right,-41.0,38.0,O,8482113.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,Anton Lundell,C,13.0,46,0.000000,0.0,0.016531,0.016531
6,2025020001,20252026,2025-10-07,FLA,CHI,3,2,851,giveaway,24,1,REG,01:10,18:50,1551,right,-13.0,-31.0,N,8484783.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,Artyom Levshunov,D,16.0,70,0.077664,1.0,0.016613,-0.061052
29,2025020001,20252026,2025-10-07,FLA,CHI,3,2,171,giveaway,66,1,REG,03:16,16:44,1551,right,14.0,-12.0,N,8473507.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,Jeff Petry,D,13.0,196,0.000000,1.0,0.015811,0.015811
61,2025020001,20252026,2025-10-07,FLA,CHI,3,2,883,giveaway,159,1,REG,09:19,10:41,1551,right,-70.0,-40.0,D,8483506.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,Sam Rinzel,D,16.0,559,0.036003,2.0,0.025382,-0.010620
94,2025020001,20252026,2025-10-07,FLA,CHI,3,2,390,giveaway,230,1,REG,14:22,05:38,1551,right,5.0,6.0,N,8483506.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,Sam Rinzel,D,16.0,862,0.000000,1.0,0.015655,0.015655
97,2025020001,20252026,2025-10-07,FLA,CHI,3,2,400,giveaway,237,1,REG,14:36,05:24,1551,right,-96.0,-22.0,D,8483506.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,Sam Rinzel,D,16.0,876,0.082346,2.0,0.027466,-0.054880
117,2025020001,20252026,2025-10-07,FLA,CHI,3,2,606,giveaway,283,1,REG,18:02,01:58,1551,right,94.0,-15.0,O,8473419.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,Brad Marchand,L,13.0,1082,0.000000,0.0,0.007104,0.007104
118,2025020001,20252026,2025-10-07,FLA,CHI,3,2,885,giveaway,284,1,REG,18:07,01:53,1551,right,31.0,28.0,O,8484783.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,Artyom Levshunov,D,16.0,1087,0.000000,0.0,0.016734,0.016734
120,2025020001,20252026,2025-10-07,FLA,CHI,3,2,608,giveaway,294,1,REG,18:37,01:23,1551,right,-76.0,-35.0,O,8476473.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,Connor Murphy,D,16.0,1117,0.000000,0.0,0.010691,0.010691
123,2025020001,20252026,2025-10-07,FLA,CHI,3,2,609,giveaway,305,1,REG,19:12,00:48,1551,right,24.0,-12.0,N,8480185.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,Eetu Luostarinen,C,13.0,1152,0.000000,1.0,0.016599,0.016599


## Summary — The Two Models

### Model 1 — xGoal Model (Shot Danger)
**Purpose:** Measure how dangerous each shot was.

**Algorithm:** Logistic Regression  
**Features:** Distance to net, angle to net, whether shot was on target  
**Target:** Did the shot become a goal? (1 or 0)  
**Performance:** AUC = 0.863 (train), 0.865 (test) — no overfitting ✅

For every shot attempt in the dataset we now have an xGoal value between 
0 and 1 representing the probability it becomes a goal. Blocked and missed 
shots receive near-zero xGoal, shots on goal scale with distance and angle.

---

### Model 2 — Above Expected Model (Giveaway Context)
**Purpose:** Learn what danger is expected from a giveaway given its context.

**Algorithm:** Random Forest Regressor  
**Features:** x/y coordinates, zone (ordinal encoded D>N>O)  
**Target:** Danger score (xGoal of next opposing shot within 15 seconds)  
**Performance:** R² = 0.022 (train), 0.013 (test) — consistent, no overfitting ✅

For every 5on5 giveaway we now have:
- `danger_score` — actual danger caused (from Model 1)
- `expected_danger` — expected danger given situation (from Model 2)
- `above_expected` — the difference

The model correctly identifies that dzone giveaways are 3x more dangerous 
than ozone giveaways on average.

---

## Metric 2- Giveaway Above Expected 

`giveaway_above_expected = expected_danger - danger_score`

**Positive value** → giveaway caused LESS danger than expected for that 
situation → good puck management

**Negative value** → giveaway caused MORE danger than expected → poor 
puck management

Where:
- `expected_danger` — what Model 2 predicts should happen given the 
  situation (zone, coordinates, period)
- `danger_score` — what actually happened (xGoal of next opposing shot 
  within 15 seconds from Model 1)


In [113]:
# Filter out goalies from giveaways before aggregating
giveaways_skaters = giveaways[giveaways['position_code'] != 'G']

# Reaggregate to player level
player_giveaways = giveaways_skaters.groupby('player_name').agg(
    total_giveaways=('above_expected', 'count'),
    avg_above_expected=('above_expected', 'mean'),
    avg_danger_score=('danger_score', 'mean'),
    avg_expected_danger=('expected_danger', 'mean')
).reset_index()

player_giveaways


,player_name,total_giveaways,avg_above_expected,avg_danger_score,avg_expected_danger
0,A.J. Greer,51,-0.000951,0.018437,0.017486
1,Aaron Ekblad,51,-0.011717,0.037052,0.025335
2,Aatu Räty,28,0.017370,0.001503,0.018873
3,Abram Wiebe,2,-0.007903,0.030445,0.022542
4,Adam Boqvist,9,-0.017522,0.041776,0.024254
...,...,...,...,...,...
909,Zack Ostapchuk,18,0.002923,0.015053,0.017975
910,Zakhar Bardakov,27,0.017652,0.000441,0.018092
911,Zayne Parekh,59,-0.007681,0.031786,0.024105
912,Zeev Buium,76,-0.006641,0.031705,0.025065


## Step 5 — Aggregate to Player Level

Now that every giveaway has an above expected score we aggregate from 
event level (one row per giveaway) to player level (one row per player).

**Process:**
1. Filter out goalies (position_code = 'G') — goalies handle the puck 
   differently and should not be included in skater puck management rankings. 
   Their giveaways almost exclusively occur right at the goal line in the 
   defensive zone which would skew the analysis.

2. Group by player name and calculate:
   - `total_giveaways` — total number of 5on5 giveaways for the season
   - `avg_above_expected` — average above expected score across all giveaways
   - `avg_danger_score` — average actual danger caused per giveaway
   - `avg_expected_danger` — average expected danger given situations

The result is 914 skaters each with a season level giveaway above 
expected score ready to be joined with our other metrics!

In [114]:
# Save to processed folder
player_giveaways.to_csv('../data/processed/player_giveaways.csv', index=False)
print(f"Saved {len(player_giveaways)} players to data/processed/player_giveaways.csv")

Saved 914 players to data/processed/player_giveaways.csv
